## Creating MCP server 
and accessing through Langchain agent 

In [1]:
import os 
from dotenv import load_dotenv 
load_dotenv() 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


import warnings

warnings.filterwarnings("ignore",category=DeprecationWarning)

from langchain.chat_models import init_chat_model

gemma = init_chat_model(model="gemma4:latest", model_provider="ollama")

#llm = init_chat_model(model="qwen/qwen3-32b", model_provider="Groq")
llm_primary = init_chat_model(model="gemma4:latest", model_provider="ollama")
llm_fallback_1 = init_chat_model(model="gpt-5.4-nano", model_provider="OpenAI")
llm_fallback_2 = init_chat_model(model="gpt-5.4-mini", model_provider="OpenAI")


In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

## Connect your client with the MongoDB-MCP-server 

In [3]:
import sys
print(sys.executable)

/Users/nali/Documents/YTLLMs/.venv/bin/python


In [6]:
client = MultiServerMCPClient({
"mcpServers": {
     "transport": "stdio",
      "command": "uv",
      "args": [
        "run",
        "python",
        "servery.py"
      ],
      "cwd": "/Users/nali/Documents/YTLLMs/AgenticAI/Self/MCP-Server"
    
  }
})

In [7]:
tools = await client.get_tools()

In [8]:
for tool in tools: 
    print(tool.name)

get_course_list
get_course_details


## Create Langchain agent with mcp_tools

In [9]:
from langchain.agents import create_agent

prompt="""You are a helpful assistant. 
use tools for answering based on user query.
"""

agent= create_agent(
    model=  llm_primary,
    tools=tools,
    system_prompt=prompt
)

## Test the agent

In [10]:
user_query= """How many courses are there and what are the course codes?
 What are the software requirements for CS 466"""

In [11]:
from langchain.messages import SystemMessage,HumanMessage

try:
    result = await agent.ainvoke({
        "messages":[
            SystemMessage(content="You a helpful assistant."),
            HumanMessage(content=user_query)
        ]
    })
except Exception as e:
    print(f"Error happened during invoke. {e}")

In [12]:
print(result["messages"][-1].content)

Based on the tools used, here is the information you requested:

### 📚 Available Courses and Codes
The following courses were found:
*   **CS 111**: AI for All - Artificial Intelligence for Life, Society, and Disciplines
*   **CS 466**: Natural Language Processing & Large Language Models (NLP & LLMs)

*(Note: The tool provided a list of these two courses.)*

### 💻 Software Requirements for CS 466
The software and tools required for **CS 466 (NLP & LLMs)** are listed under the "Materials and Tools" section:

*   zyBook for CS 466
*   VS Code
*   Python 3.x
*   PyTorch
*   Hugging Face Transformers and Datasets
*   LangChain and LangGraph
*   SentenceTransformers
*   FAISS or ChromaDB
*   Local GPU, Google Colab, Kaggle, or university computing resources
